In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.neural_network import MLPRegressor
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
import pandas as pd

train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
df = train
submission = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv')
print(f"Train shape: {train.shape}, Test shape: {test.shape}")

Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(train['SalePrice'], kde=True)
plt.title('SalePrice Distribution')
plt.show()

Correlation Heatmap

In [ ]:
# Only use numeric columns for correlation
corrmat = train.select_dtypes(include=[np.number]).corr()
top_corr = corrmat.nlargest(15, 'SalePrice')['SalePrice'].index

plt.figure(figsize=(10,8))
sns.heatmap(train[top_corr].corr(), annot=True, cmap='coolwarm')
plt.title('Top 15 Feature Correlations with SalePrice')
plt.show()
# Missing values
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
plt.figure(figsize=(8,5))
sns.barplot(x=missing.values, y=missing.index)
plt.title('Missing Values (Train Set)')
plt.xlabel('Count')
plt.show()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
# Impute missing values based on AI suggestion
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
df[categorical_cols] = df[categorical_cols].fillna(df[categorical_cols].mode().iloc[0])

# Outlier handling: cap at 1st and 99th percentile
for col in numeric_cols:
    df[col] = df[col].clip(df[col].quantile(0.01), df[col].quantile(0.99))

Data Cleaning and Feature Engineering

In [ ]:

# Concatenate train & test for processing
y = train['SalePrice']
train_ID = train['Id']
test_ID = test['Id']
full = pd.concat([train.drop(['SalePrice'], axis=1), test], axis=0, ignore_index=True)
print(f"Full dataset shape: {full.shape}")
# Handle missing values and feature types
for col in full.columns:
    if full[col].dtype == "object":
        full[col] = full[col].fillna('None')
    else:
        full[col] = full[col].fillna(full[col].median())

# Label encoding for categorical variables
for col in full.select_dtypes(include='object').columns:
    lbl = LabelEncoder()
    full[col] = lbl.fit_transform(full[col].astype(str))

Remove outliers (optional, based on EDA)


In [ ]:
# Remove outliers in GrLivArea (living area)
train_index = range(train.shape[0])
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index
full.drop(outliers, inplace=True)
y = y.drop(outliers)

In [ ]:
# Feature transformation (log transform target & skewed features)
y = np.log1p(y)

# Log transform skewed numeric features
from scipy.stats import skew

# Only select numeric columns
numeric_feats = full.select_dtypes(include=[np.number]).columns

# Calculate skewness on the numeric features of the processed dataset
skewed_feats = full[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)

# Only select features with high skewness for transformation
skewness = skewed_feats[abs(skewed_feats) > 0.75]
print(f"Skewed features:\n{skewness}")

# Apply log1p transformation to those features
for feat in skewness.index:
    full[feat] = np.log1p(full[feat])

# Split
X = full.iloc[:len(y), :]
X_test = full.iloc[len(y):, :]

Model Training and Evaluation

In [ ]:
# a. Train-validation split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)


# b. Define and train models

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42),
    'CatBoost': cb.CatBoostRegressor(iterations=100, random_state=42, verbose=0),
    'Neural Net': MLPRegressor(random_state=0, max_iter=300)

}
rmse_dict = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    rmse_dict[name] = rmse
    print(f"{name} RMSE: {rmse:.5f}")


### c. RMSE Comparison Table

rmse_table = pd.DataFrame({'Model': list(rmse_dict.keys()), 'RMSE': list(rmse_dict.values())}).sort_values('RMSE')
print(rmse_table)


Feature Importance Visualizations

In [ ]:
best_model_name = rmse_table.iloc[0]['Model']
best_model = models[best_model_name]

# For tree-based models, plot feature importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[-10:][::-1]
    plt.figure(figsize=(10,5))
    plt.title(f'Top 10 Feature Importances ({best_model_name})')
    plt.bar(range(10), importances[indices], color='b', align='center')
    plt.xticks(range(10), X.columns[indices], rotation=90)
    plt.show()
else:
    print(f"{best_model_name} does not support feature_importances_.")

Predict and Generate Submission File

In [ ]:
# Retrain best model on all training data
best_model.fit(X, y)
final_preds = best_model.predict(X_test)
final_preds = np.expm1(final_preds)  # Inverse log1p

submission['SalePrice'] = final_preds
submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")

Data Visualization

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x=train['OverallQual'], y=train['SalePrice'])
plt.title('SalePrice vs OverallQual')
plt.show()

# SalePrice vs. GrLivArea
plt.figure(figsize=(8,5))
sns.scatterplot(x=train['GrLivArea'], y=train['SalePrice'])
plt.title('SalePrice vs GrLivArea')
plt.show()

In [ ]:
# Residual plot
val_preds = best_model.predict(X_valid)
residuals = y_valid - val_preds
plt.figure(figsize=(8,4))
plt.scatter(val_preds, residuals, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

# Actual vs Predicted
plt.figure(figsize=(8,4))
plt.scatter(y_valid, val_preds, alpha=0.3)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()